In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# -------------------------------
# 1. Download Stock Market Data
# -------------------------------

ticker = "RELIANCE.NS"

data = yf.download(
    ticker,
    period="5y",
    interval="1d",
    auto_adjust=True,
    progress=False
)

if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

data = data[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

# -------------------------------
# 2. Calculate Technical Indicators
# -------------------------------

data['SMA'] = data['Close'].rolling(14).mean()

data['WMA'] = data['Close'].rolling(14).apply(
    lambda x: np.dot(x, np.arange(1, 15)) / 105,
    raw=True
)

delta = data['Close'].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss.replace(0, np.nan)

data['RSI'] = 100 - (100 / (1 + rs))

low14 = data['Low'].rolling(14).min()
high14 = data['High'].rolling(14).max()

data['Stochastic_K'] = (
    100 * (data['Close'] - low14) /
    (high14 - low14).replace(0, np.nan)
)

data['Stochastic_D'] = data['Stochastic_K'].rolling(3).mean()

# Next day's closing price
data['Target'] = data['Close'].shift(-1)

data.dropna(inplace=True)

features = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'SMA', 'WMA', 'RSI',
    'Stochastic_K', 'Stochastic_D'
]

X = data[features]
y = data['Target']

# -------------------------------
# 3. Split Dataset Chronologically
# -------------------------------

n = len(data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end]
y_train = y.iloc[:train_end]

X_val = X.iloc[train_end:val_end]
y_val = y.iloc[train_end:val_end]

X_test = X.iloc[val_end:]
y_test = y.iloc[val_end:]

# -------------------------------
# 4. Genetic Algorithm Settings
# -------------------------------

POPULATION_SIZE = 20
GENERATIONS = 15
CROSSOVER_RATE = 0.8
MUTATION_RATE = 0.1

NUM_FEATURES = len(features)

random.seed(42)
np.random.seed(42)

# Generate initial population
population = [
    [random.randint(0, 1) for _ in range(NUM_FEATURES)]
    for _ in range(POPULATION_SIZE)
]

# Ensure at least one feature is selected
for chromosome in population:
    if sum(chromosome) == 0:
        chromosome[random.randrange(NUM_FEATURES)] = 1

# -------------------------------
# 5. Fitness Function
# -------------------------------

def fitness(chromosome):

    selected = [
        features[i]
        for i in range(NUM_FEATURES)
        if chromosome[i] == 1
    ]

    if len(selected) == 0:
        return float('inf')

    model = RandomForestRegressor(
        n_estimators=50,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train[selected], y_train)

    predictions = model.predict(X_val[selected])

    rmse = np.sqrt(
        mean_squared_error(y_val, predictions)
    )

    # Penalize excessive feature selection
    penalty = 0.01 * (len(selected) / NUM_FEATURES)

    return rmse * (1 + penalty)


# -------------------------------
# 6. Selection Operation
# -------------------------------

def selection(population):

    candidates = random.sample(population, 3)

    best = min(candidates, key=fitness)

    return best.copy()


# -------------------------------
# 7. Crossover Operation
# -------------------------------

def crossover(parent1, parent2):

    if random.random() < CROSSOVER_RATE:

        point = random.randint(1, NUM_FEATURES - 1)

        child1 = parent1[:point] + parent2[point:]
        child2 = parent2[:point] + parent1[point:]

        return child1, child2

    return parent1.copy(), parent2.copy()


# -------------------------------
# 8. Mutation Operation
# -------------------------------

def mutation(chromosome):

    for i in range(NUM_FEATURES):

        if random.random() < MUTATION_RATE:
            chromosome[i] = 1 - chromosome[i]

    if sum(chromosome) == 0:
        chromosome[random.randrange(NUM_FEATURES)] = 1

    return chromosome


# -------------------------------
# 9. Execute Genetic Algorithm
# -------------------------------

best_fitness_history = []

for generation in range(GENERATIONS):

    scores = [fitness(c) for c in population]

    best_index = np.argmin(scores)

    best_chromosome = population[best_index].copy()

    best_fitness_history.append(scores[best_index])

    print(
        "Generation:", generation + 1,
        "| Best RMSE:", round(scores[best_index], 4),
        "| Selected Features:", sum(best_chromosome)
    )

    new_population = [best_chromosome.copy()]

    while len(new_population) < POPULATION_SIZE:

        parent1 = selection(population)
        parent2 = selection(population)

        child1, child2 = crossover(parent1, parent2)

        child1 = mutation(child1)
        child2 = mutation(child2)

        new_population.append(child1)

        if len(new_population) < POPULATION_SIZE:
            new_population.append(child2)

    population = new_population

# -------------------------------
# 10. Obtain Best Features
# -------------------------------

best_chromosome = min(population, key=fitness)

selected_features = [
    features[i]
    for i in range(NUM_FEATURES)
    if best_chromosome[i] == 1
]

print("\nBest Selected Features:")
print(selected_features)

# -------------------------------
# 11. Train Final Model
# -------------------------------

X_train_final = X.iloc[:val_end][selected_features]
y_train_final = y.iloc[:val_end]

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_final, y_train_final)

# -------------------------------
# 12. Test Prediction
# -------------------------------

predictions = model.predict(X_test[selected_features])

rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("\n----- MODEL PERFORMANCE -----")
print("RMSE:", round(rmse, 2))
print("MAE:", round(mae, 2))
print("R2 Score:", round(r2, 4))

# -------------------------------
# 13. Next Day Stock Prediction
# -------------------------------

latest_data = yf.download(
    ticker,
    period="1y",
    interval="1d",
    auto_adjust=True,
    progress=False
)

if isinstance(latest_data.columns, pd.MultiIndex):
    latest_data.columns = latest_data.columns.get_level_values(0)

latest_data = latest_data[
    ['Open', 'High', 'Low', 'Close', 'Volume']
].copy()

latest_data['SMA'] = latest_data['Close'].rolling(14).mean()

latest_data['WMA'] = latest_data['Close'].rolling(14).apply(
    lambda x: np.dot(x, np.arange(1, 15)) / 105,
    raw=True
)

delta = latest_data['Close'].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss.replace(0, np.nan)

latest_data['RSI'] = 100 - (100 / (1 + rs))

low14 = latest_data['Low'].rolling(14).min()
high14 = latest_data['High'].rolling(14).max()

latest_data['Stochastic_K'] = (
    100 * (latest_data['Close'] - low14) /
    (high14 - low14).replace(0, np.nan)
)

latest_data['Stochastic_D'] = (
    latest_data['Stochastic_K'].rolling(3).mean()
)

latest_data.dropna(inplace=True)

latest_row = latest_data[selected_features].iloc[[-1]]

next_price = model.predict(latest_row)[0]

current_price = latest_data['Close'].iloc[-1]

print("\n----- FINANCIAL PREDICTION -----")
print("Stock:", ticker)
print("Latest Closing Price:", round(current_price, 2))
print("Predicted Next Closing Price:", round(next_price, 2))

if next_price > current_price:
    print("Predicted Trend: UPWARD")
elif next_price < current_price:
    print("Predicted Trend: DOWNWARD")
else:
    print("Predicted Trend: STABLE")

# -------------------------------
# 14. Visualization
# -------------------------------

plt.figure(figsize=(10, 5))

plt.plot(
    y_test.index,
    y_test.values,
    label="Actual Price"
)

plt.plot(
    y_test.index,
    predictions,
    label="Predicted Price"
)

plt.title("Stock Price Prediction using GA")
plt.xlabel("Date")
plt.ylabel("Stock Price")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

# GA Convergence Graph

plt.figure(figsize=(8, 4))

plt.plot(
    range(1, GENERATIONS + 1),
    best_fitness_history,
    marker='o'
)

plt.title("Genetic Algorithm Convergence")
plt.xlabel("Generation")
plt.ylabel("Best RMSE")
plt.grid()
plt.tight_layout()
plt.show()


Generation: 1 | Best RMSE: 19.772 | Selected Features: 4
Generation: 2 | Best RMSE: 19.2855 | Selected Features: 5
Generation: 3 | Best RMSE: 18.3614 | Selected Features: 4
Generation: 4 | Best RMSE: 18.1615 | Selected Features: 2
Generation: 5 | Best RMSE: 18.0179 | Selected Features: 3
Generation: 6 | Best RMSE: 18.0179 | Selected Features: 3
Generation: 7 | Best RMSE: 18.0179 | Selected Features: 3
Generation: 8 | Best RMSE: 18.0179 | Selected Features: 3
Generation: 9 | Best RMSE: 18.0179 | Selected Features: 3
